In [ ]:
import pandas as pd
import numpy as np
import sqlite3

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
conn = sqlite3.connect('../data/ecommerce.db')

In [ ]:
query = """
SELECT 
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    p.payment_value
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
JOIN payments p
ON o.order_id = p.order_id
"""

df = pd.read_sql(query, conn)

df.head()

In [ ]:
df['order_purchase_timestamp'] = pd.to_datetime(
    df['order_purchase_timestamp']
)

In [ ]:
reference_date = df['order_purchase_timestamp'].max()

Create RFM Data

In [ ]:
rfm = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - x.max()).days,
    'order_id': 'count',
    'payment_value': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

In [ ]:
rfm.describe()

In [ ]:
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)

In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

rfm.head()

In [ ]:
rfm['Cluster'].value_counts()

In [ ]:
cluster_summary = rfm.groupby('Cluster').mean()

cluster_summary

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=rfm,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2'
)

plt.title('Customer Segmentation')

plt.show()

## Customer Segment Insights

### Cluster 0
- High spending customers
- Frequent purchases
- Loyal customer base

### Cluster 1
- Low frequency customers
- Potential churn risk

### Cluster 2
- New or occasional customers

### Cluster 3
- Medium-value regular customers

In [ ]:
import joblib

joblib.dump(kmeans, '../models/kmeans_model.pkl')